# 01 - Agent 控制循环 (Agent Control Loop)

## 学习目标

1. 理解 Agent 的核心架构：**Agent = LLM + Tools + Memory + Control Loop**
2. 实现完整的 `BaseAgent` 基类，包含状态机和主循环
3. 掌握 Perception（感知）→ Planning（规划）→ Action（行动）→ Reflection（反思）循环
4. 理解状态转移和 `max_iterations` 防护机制
5. 用 Mock LLM 和工具展示完整的 Agent 执行流程

## Agent 架构概览

```
                    +-----------+
                    |   用户    |
                    +-----+-----+
                          |
                          v (任务)
     +------------------- Agent -------------------+
     |                                              |
     |  +--------+    +--------+    +--------+      |
     |  |  LLM   |<-->| Memory |<-->| Tools  |      |
     |  +--------+    +--------+    +--------+      |
     |       ^                        ^             |
     |       |                        |             |
     |  +----+------------------------+----+        |
     |  |        Control Loop              |        |
     |  |  Perceive → Plan → Act → Reflect |        |
     |  +----------------------------------+        |
     |                                              |
     +----------------------------------------------+
                          |
                          v (回答)
                    +-----+-----+
                    |   用户    |
                    +-----------+
```

In [ ]:
# 核心导入
from dataclasses import dataclass, field
from typing import Any, Callable, Optional
from enum import Enum
from abc import ABC, abstractmethod
import time
import json

## 1. Agent 状态机 (AgentState)

Agent 在执行过程中经历多个状态。每个状态定义了 Agent 当前在做什么。

状态转移图：

```
                    +-------+
                    | IDLE  |  (初始状态)
                    +---+---+
                        |
                        v
                    +-----------+       +-----------+
                    | PERCEIVING|------>|   ERROR   |
                    +-----+-----+       +-----------+
                          |                   ^
                          v                   |
                    +----------+              |
                    | PLANNING |------------->+
                    +-----+----+              |
                          |                   |
                          v                   |
                    +---------+               |
                    | ACTING  |-------------->+
                    +----+----+               |
                         |                    |
                         v                    |
                    +------------+            |
               +----| REFLECTING |<-------+   |
               |    +-----+------+        |   |
               |          |               |   |
               |          v               |   |
               |    (need_more?) ---Yes---+   |
               |          |                    |
               |         No                    |
               |          |                    |
               |          v                    |
               |    +----------+               |
               +--->| FINISHED |               |
                    +----------+               |
```

**状态说明**:
- `IDLE`: 初始状态，等待任务
- `PERCEIVING`: 解析用户输入，收集上下文信息
- `PLANNING`: 制定行动计划（分解任务、选择工具）
- `ACTING`: 执行计划中的步骤（调用工具）
- `REFLECTING`: 评估结果，决定是否继续
- `FINISHED`: 任务完成，准备返回答案
- `ERROR`: 发生错误，需要异常处理

In [ ]:
class AgentState(Enum):
    """Agent 状态枚举。

    定义了 Agent 在控制循环中可能处于的所有状态。
    每个状态决定 Agent 下一步应该执行什么操作。
    """
    IDLE = "idle"               # 空闲，等待任务
    PERCEIVING = "perceiving"   # 感知：解析用户输入
    PLANNING = "planning"       # 规划：制定行动计划
    ACTING = "acting"           # 行动：执行工具调用
    REFLECTING = "reflecting"   # 反思：评估结果，决定是否继续
    FINISHED = "finished"       # 完成：返回最终答案
    ERROR = "error"             # 错误：需要处理异常

    def __str__(self):
        """中文状态名称用于可视化。"""
        names = {
            "idle": "空闲",
            "perceiving": "感知",
            "planning": "规划",
            "acting": "行动",
            "reflecting": "反思",
            "finished": "完成",
            "error": "错误",
        }
        return names.get(self.value, self.value)


# 演示状态枚举
for state in AgentState:
    print(f"  {state.name:15s} -> {state.value:15s} -> {str(state)}")

## 2. AgentContext 数据类

AgentContext 承载 Agent 在整个执行过程中的所有状态数据。
它在各阶段之间传递，每个阶段都可以读取和修改。

In [ ]:
@dataclass
class AgentContext:
    """Agent 执行上下文 —— 在整个控制循环中传递的状态数据。

    这是 Agent 的"工作记忆"：每次循环迭代中，Agent 读取上下文
    来决定下一步该做什么，然后更新上下文来反映最新的状态。

    Attributes:
        task: 用户的原始任务描述
        history: 对话历史（用户-Agent 交互记录）
        observations: 工具调用的返回结果列表
        plan: 当前行动计划（子任务列表）
        current_step: 当前执行步数
        max_iterations: 最大迭代次数（防止无限循环）
        state: 当前 Agent 状态
    """
    task: str                                    # 原始任务
    history: list = field(default_factory=list)  # 对话历史
    observations: list = field(default_factory=list)  # 工具观察结果
    plan: list = field(default_factory=list)     # 行动计划
    current_step: int = 0                        # 当前步数
    max_iterations: int = 15                     # 最大迭代次数
    state: AgentState = AgentState.IDLE          # 当前状态

    def add_observation(self, observation: str):
        """添加工具观察结果。"""
        self.observations.append({
            "step": self.current_step,
            "content": observation,
            "timestamp": time.time(),
        })

    def add_to_history(self, role: str, content: str):
        """添加对话历史条目。"""
        self.history.append({
            "role": role,
            "content": content,
            "step": self.current_step,
        })

    def has_reached_limit(self) -> bool:
        """检查是否已达到最大迭代次数。"""
        return self.current_step >= self.max_iterations

    def summary(self) -> str:
        """生成上下文的简洁摘要（用于调试）。"""
        return (
            f"AgentContext(step={self.current_step}/{self.max_iterations}, "
            f"state={self.state}, "
            f"plan_items={len(self.plan)}, "
            f"observations={len(self.observations)}, "
            f"history_entries={len(self.history)})"
        )


# 演示上下文创建
ctx = AgentContext(
    task="查询北京天气并给出穿衣建议",
    max_iterations=10
)
print(f"初始上下文: {ctx.summary()}")
ctx.add_to_history("user", "今天北京天气怎么样？")
ctx.add_observation("北京今天晴天，22°C")
print(f"添加数据后: {ctx.summary()}")

## 3. BaseAgent 基类

抽象基类定义了 Agent 的骨架：状态机 + 控制循环。
具体的 Agent（如 ReActAgent）继承此类并实现具体的方法。

### 核心控制流程

```
run(task)
   |
   v
IDLE → PERCEIVING → PLANNING → ACTING → REFLECTING
                                            |
                              +---Yes--- (need_more?)
                              |             |
                              |            No
                              |             |
                              +<---- FINISHED
```

In [ ]:
class BaseAgent(ABC):
    """Agent 抽象基类。

    实现 Agent 的核心控制循环：
    Perception（感知）→ Planning（规划）→ Action（行动）→ Reflection（反思）

    这个基类定义了 Agent 的骨架，具体 Agent 实现继承此类
    并实现各个抽象方法。这种设计使得不同的 Agent 模式
    （ReAct, Plan-Execute, Reflection）可以共享核心循环逻辑。

    Attributes:
        llm: 语言模型客户端
        tools: 可用工具字典 {名称: 可调用对象}
        memory: 记忆系统（短期 + 长期）
        max_iterations: 最大迭代次数
        context: 当前执行上下文
    """

    def __init__(self, llm, tools: dict, memory=None, max_iterations: int = 15):
        """初始化 Agent。

        Args:
            llm: 语言模型客户端（需有 generate 或 __call__ 方法）
            tools: 工具字典 {工具名称: 可调用对象}
            memory: 记忆系统（可选，默认使用列表作为短期记忆）
            max_iterations: 最大迭代次数（防止无限循环的安全阀）
        """
        self.llm = llm
        self.tools = tools
        self.memory = memory if memory is not None else []
        self.max_iterations = max_iterations
        self.context: Optional[AgentContext] = None
        self._state_history: list = []  # 记录状态转移路径

    def _transition_to(self, new_state: AgentState):
        """执行状态转移并记录。

        状态转移是 Agent 执行的核心机制。
        每次转移都被记录以便调试和可视化。

        Args:
            new_state: 目标状态
        """
        old_state = self.context.state if self.context else AgentState.IDLE
        self.context.state = new_state
        self._state_history.append({
            "step": self.context.current_step if self.context else 0,
            "from": old_state.value,
            "to": new_state.value,
        })

    # ===== 抽象方法 —— 子类必须实现 =====

    @abstractmethod
    def perceive(self, context: AgentContext) -> dict:
        """感知阶段：解析用户输入，提取关键信息。

        此方法分析用户的任务，提取实体、意图、约束等信息。
        返回结构化的感知结果供规划阶段使用。

        Args:
            context: 当前执行上下文

        Returns:
            结构化感知结果，例如:
            {
                "intent": "weather_query",
                "entities": ["北京", "今天"],
                "constraints": ["给出穿衣建议"],
            }
        """
        ...

    @abstractmethod
    def plan(self, context: AgentContext) -> list:
        """规划阶段：制定行动计划。

        基于感知结果，制定一个多步骤的行动计划。
        每个计划项是一个 (tool_name, params) 元组。

        Args:
            context: 当前执行上下文（包含感知结果）

        Returns:
            行动计划列表，例如:
            [
                ("get_weather", {"city": "北京", "date": "today"}),
                ("web_search", {"query": "22度穿什么衣服"}),
            ]
        """
        ...

    @abstractmethod
    def act(self, context: AgentContext, action_plan: list) -> dict:
        """行动阶段：执行行动计划。

        依次（或并行）执行计划中的每个步骤。
        收集所有工具调用的结果。

        Args:
            context: 当前执行上下文
            action_plan: 行动计划（来自 plan 方法的输出）

        Returns:
            执行结果字典:
            {
                "success": True/False,
                "results": [{"tool": "...", "result": "..."}, ...],
                "errors": [...],
            }
        """
        ...

    @abstractmethod
    def reflect(self, context: AgentContext, result: dict) -> bool:
        """反思阶段：评估执行结果，决定是否继续。

        分析行动结果的质量，判断是否:
        1. 已经足够回答用户 → 返回 False（结束循环）
        2. 还需要更多信息 → 返回 True（继续循环）
        3. 需要调整计划 → 修改 context.plan 并返回 True

        Args:
            context: 当前执行上下文
            result: 行动阶段的执行结果

        Returns:
            True 表示需要继续循环，False 表示可以结束
        """
        ...

    # ===== 主控制循环 =====

    def run(self, task: str) -> AgentContext:
        """主控制循环 —— Agent 的核心执行入口。

        这是 Agent 的"心跳"：从 IDLE 开始，循环执行
        Perceive → Plan → Act → Reflect，直到任务完成
        或达到最大迭代次数。

        流程伪代码:
        ```
        context = AgentContext(task)
        context.state = PERCEIVING
        while context.current_step < max_iterations:
            switch context.state:
                case PERCEIVING:
                    perception = self.perceive(context)
                    context.state = PLANNING
                case PLANNING:
                    context.plan = self.plan(context)
                    context.state = ACTING
                case ACTING:
                    result = self.act(context, context.plan)
                    context.state = REFLECTING
                case REFLECTING:
                    if not self.reflect(context, result):
                        context.state = FINISHED
                        break
                    context.state = PLANNING  # 重新规划
                case FINISHED:
                    break
                case ERROR:
                    self._handle_error(context)
                    break
            context.current_step += 1
        return context
        ```

        Args:
            task: 用户任务描述

        Returns:
            AgentContext 包含完整的执行记录
        """
        # 初始化上下文
        self.context = AgentContext(
            task=task,
            max_iterations=self.max_iterations,
        )
        self._state_history = []
        self.context.add_to_history("user", task)

        # 开始控制循环
        self._transition_to(AgentState.PERCEIVING)

        while not self.context.has_reached_limit():
            try:
                if self.context.state == AgentState.PERCEIVING:
                    self._step_perceive()

                elif self.context.state == AgentState.PLANNING:
                    self._step_plan()

                elif self.context.state == AgentState.ACTING:
                    self._step_act()

                elif self.context.state == AgentState.REFLECTING:
                    should_continue = self._step_reflect()
                    if not should_continue:
                        self._transition_to(AgentState.FINISHED)
                        break

                elif self.context.state == AgentState.FINISHED:
                    break

                elif self.context.state == AgentState.ERROR:
                    self._handle_error(self.context)
                    break

                else:
                    # 未知状态，回退到 PERCEIVING
                    self._transition_to(AgentState.PERCEIVING)

                self.context.current_step += 1

            except Exception as e:
                print(f"[异常] Step {self.context.current_step}: {str(e)}")
                self._transition_to(AgentState.ERROR)
                self.context.add_to_history("system", f"Error: {str(e)}")
                break

        # 达到最大迭代次数
        if self.context.has_reached_limit() and self.context.state != AgentState.FINISHED:
            print(f"[警告] 达到最大迭代次数 ({self.max_iterations})，强制终止")
            self._transition_to(AgentState.FINISHED)

        return self.context

    def _step_perceive(self):
        """执行感知步骤。"""
        print(f"  [感知] Step {self.context.current_step}: 分析用户任务...")
        perception = self.perceive(self.context)
        self.context.add_to_history("perception", json.dumps(perception, ensure_ascii=False))
        self._transition_to(AgentState.PLANNING)

    def _step_plan(self):
        """执行规划步骤。"""
        print(f"  [规划] Step {self.context.current_step}: 制定行动计划...")
        self.context.plan = self.plan(self.context)
        self._transition_to(AgentState.ACTING)

    def _step_act(self):
        """执行行动步骤。"""
        print(f"  [行动] Step {self.context.current_step}: 执行计划 (共 {len(self.context.plan)} 项)...")
        result = self.act(self.context, self.context.plan)
        self.context.add_to_history("action_result", json.dumps(result, ensure_ascii=False))
        # 存储结果供反思阶段使用
        self._last_action_result = result
        self._transition_to(AgentState.REFLECTING)

    def _step_reflect(self) -> bool:
        """执行反思步骤。

        Returns:
            True 如果还需要继续循环，False 如果可以结束
        """
        print(f"  [反思] Step {self.context.current_step}: 评估结果...")
        result = getattr(self, '_last_action_result', {})
        should_continue = self.reflect(self.context, result)
        if should_continue:
            self._transition_to(AgentState.PLANNING)
        return should_continue

    def _handle_error(self, context: AgentContext):
        """错误处理逻辑（子类可覆盖）。"""
        print(f"[错误处理] Agent 在 Step {context.current_step} 发生错误")
        print(f"  状态历史: {self._state_history}")

    def print_state_history(self):
        """打印状态转移历史（用于可视化）。"""
        if not self._state_history:
            print("无状态转移记录")
            return

        print("\n状态转移历史:")
        print("-" * 50)
        for entry in self._state_history:
            step = entry["step"]
            from_state = entry["from"]
            to_state = entry["to"]
            print(f"  Step {step}: {from_state:15s} → {to_state}")
        print("-" * 50)

## 4. 具体 Agent 实现：SimpleAgent

现在实现一个具体的 Agent 子类，将抽象方法填充为实际逻辑。
这个 SimpleAgent 使用 Mock LLM 和 Mock 工具来演示完整的控制循环。

In [ ]:
class SimpleAgent(BaseAgent):
    """简单的具体 Agent 实现。

    继承了 BaseAgent 的控制循环骨架，
    实现了 perceive / plan / act / reflect 的具体逻辑。

    这个实现使用规则（而非 LLM）来做决策，
    目的是清晰地展示控制循环的流程。
    """

    def __init__(self, llm, tools: dict, memory=None, max_iterations: int = 15):
        super().__init__(llm, tools, memory, max_iterations)
        self.final_answer: str = ""

    def perceive(self, context: AgentContext) -> dict:
        """感知：从用户任务中提取意图和实体。

        使用简单的关键词匹配来模拟 LLM 的感知能力。
        实际生产环境中，这里会调用 LLM 来理解用户意图。
        """
        task = context.task
        task_lower = task.lower()

        perception = {
            "intent": "unknown",
            "entities": [],
            "requires_tools": [],
        }

        # 意图识别（基于关键词）
        if any(kw in task_lower for kw in ["天气", "weather", "气温"]):
            perception["intent"] = "weather_query"
            perception["requires_tools"].append("get_weather")
            # 提取城市名
            for city in ["北京", "上海", "广州", "深圳", "巴黎", "东京", "纽约"]:
                if city in task:
                    perception["entities"].append(city)

        if any(kw in task_lower for kw in ["计算", "算", "+", "-", "*", "/"]):
            perception["intent"] = "calculation"
            perception["requires_tools"].append("calculator")

        if any(kw in task_lower for kw in ["搜索", "查询", "什么是", "介绍", "门票"]):
            if perception["intent"] == "unknown":
                perception["intent"] = "information_search"
            perception["requires_tools"].append("web_search")

        if any(kw in task_lower for kw in ["员工", "部门", "工资", "薪资"]):
            perception["intent"] = "database_query"
            perception["requires_tools"].append("database_query")

        if any(kw in task_lower for kw in ["文件", "读取"]):
            perception["intent"] = "file_read"
            perception["requires_tools"].append("file_reader")

        return perception

    def plan(self, context: AgentContext) -> list:
        """规划：基于感知结果制定行动计划。

        将感知到的需求映射为具体的工具调用计划。
        """
        perception = None
        for entry in context.history:
            if entry.get("role") == "perception":
                try:
                    perception = json.loads(entry["content"])
                except (json.JSONDecodeError, TypeError):
                    pass

        if not perception:
            return []

        plan = []
        intent = perception.get("intent", "unknown")
        entities = perception.get("entities", [])
        task = context.task

        if intent == "weather_query" and entities:
            city = entities[0]
            plan.append(("get_weather", {"city": city, "date": "today"}))
            # 如果还要求穿衣建议，添加搜索
            if any(kw in task for kw in ["穿衣", "穿什么", "建议"]):
                plan.append(("web_search", {"query": f"{city} {22}度 穿衣建议"}))

        elif intent == "calculation":
            # 从任务中提取表达式
            import re
            expr_match = re.search(r'[\d\+\-\*\/\s\(\)\.]+', task)
            if expr_match:
                expr = expr_match.group().strip()
                plan.append(("calculator", {"expression": expr}))

        elif intent == "information_search":
            search_query = task.replace("查询", "").replace("搜索", "").strip()
            plan.append(("web_search", {"query": search_query[:100]}))

        elif intent == "database_query":
            plan.append(("database_query", {"sql": "SELECT * FROM employees"}))

        elif intent == "file_read":
            plan.append(("file_reader", {"filepath": task.split()[-1] if task.split() else "unknown"}))

        if not plan:
            plan.append(("web_search", {"query": task[:100]}))

        return plan

    def act(self, context: AgentContext, action_plan: list) -> dict:
        """行动：执行计划中的每个步骤。

        依次调用每个工具，收集所有结果。
        错误被捕获但不会中断后续步骤。
        """
        results = []
        errors = []

        for tool_name, params in action_plan:
            print(f"    -> 调用工具: {tool_name}({params})")

            if tool_name not in self.tools:
                errors.append(f"工具 '{tool_name}' 不存在")
                continue

            try:
                tool_func = self.tools[tool_name]
                raw_result = tool_func(**params)
                result_str = str(raw_result)

                # 截断过长的结果
                if len(result_str) > 500:
                    result_str = result_str[:500] + "..."

                results.append({
                    "tool": tool_name,
                    "params": params,
                    "result": result_str,
                })

                context.add_observation(result_str)
                print(f"    <- 结果: {result_str[:100]}...")

            except TypeError as e:
                errors.append(f"{tool_name}: 参数错误 - {str(e)}")
            except Exception as e:
                errors.append(f"{tool_name}: 执行异常 - {str(e)}")

        return {
            "success": len(errors) == 0,
            "results": results,
            "errors": errors,
            "total_tools_called": len(results),
        }

    def reflect(self, context: AgentContext, result: dict) -> bool:
        """反思：评估结果是否足以回答问题。

        判断标准:
        1. 有工具返回了有效结果 → 足够
        2. 所有工具都失败了 → 不够，但也不再继续（避免无限循环）
        3. 已经执行了多次循环 → 强制结束
        """
        results = result.get("results", [])
        errors = result.get("errors", [])

        # 有成功的结果 → 可以结束
        if results:
            print(f"    反思: 获得了 {len(results)} 个工具结果，信息充足，准备回答")
            self.final_answer = self._generate_answer(context, results)
            return False  # 不需要继续

        # 有错误但没有结果 → 尝试重试一次
        if errors and context.current_step < 2:
            print(f"    反思: 工具调用失败 ({len(errors)} 个错误)，尝试重新规划")
            return True  # 继续循环

        # 已尝试多次 → 不再继续
        print(f"    反思: 经过 {context.current_step + 1} 次尝试，无法完成任务")
        self.final_answer = f"抱歉，无法完成任务。错误: {'; '.join(errors)}"
        return False

    def _generate_answer(self, context: AgentContext, results: list) -> str:
        """基于工具结果生成最终回答。

        实际生产环境中，这里会调用 LLM 来生成自然语言回答。
        这里使用简单的模板拼接。
        """
        task = context.task
        answer_parts = [f"关于 '{task}' 的查询结果如下："]

        for r in results:
            answer_parts.append(f"\n[{r['tool']}] {r['result']}")

        return "\n".join(answer_parts)

## 5. Mock 工具和 LLM

为了完整演示控制循环，我们需要一组 Mock 工具和一个 Mock LLM。
这些模拟组件让我们可以在不依赖外部 API 的情况下观察 Agent 的行为。

In [ ]:
# ===== Mock 工具集 =====

def mock_get_weather(city: str, date: str = "today") -> str:
    """模拟天气查询。"""
    weather_db = {
        "北京": "晴天，22°C，湿度45%，北风3级",
        "上海": "阴天，25°C，湿度70%",
        "广州": "雷阵雨，28°C，湿度85%",
        "巴黎": "晴天，20°C，湿度50%",
    }
    weather = weather_db.get(city, f"{city}今天多云，18°C（模拟数据）")
    return f"{city}天气({date}): {weather}"


def mock_calculator(expression: str) -> str:
    """模拟安全计算器。"""
    try:
        import ast
        import operator as op
        allowed = {
            ast.Add: op.add, ast.Sub: op.sub,
            ast.Mult: op.mul, ast.Div: op.truediv,
        }
        def _eval(node):
            if isinstance(node, ast.Constant):
                return node.value
            elif isinstance(node, ast.BinOp):
                return allowed[type(node.op)](_eval(node.left), _eval(node.right))
            elif isinstance(node, ast.Expression):
                return _eval(node.body)
            raise ValueError("不支持")
        tree = ast.parse(expression.strip(), mode="eval")
        result = _eval(tree)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"


def mock_web_search(query: str) -> str:
    """模拟网络搜索。"""
    results = {
        "故宫": "故宫博物院：旺季60元，淡季40元，周一闭馆",
        "python": "Python 是一种高级编程语言",
        "穿衣": "22°C建议穿薄外套或长袖T恤",
    }
    for key, val in results.items():
        if key in query:
            return f"搜索结果: {val}"
    return f"关于 '{query}' 的搜索结果（模拟数据）"


def mock_database_query(sql: str) -> str:
    """模拟数据库查询。"""
    if "SELECT" not in sql.upper():
        return "错误：仅支持 SELECT"
    return "查询结果: [{'id':1, 'name':'张三', 'department':'技术部', 'salary':25000}]"


def mock_file_reader(filepath: str) -> str:
    """模拟文件读取。"""
    import os
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            content = f.read()[:500]
        return f"文件内容: {content}"
    return f"模拟文件内容（{filepath}）: 这是一个演示文件。"


# 收集所有工具
MOCK_TOOLS = {
    "get_weather": mock_get_weather,
    "calculator": mock_calculator,
    "web_search": mock_web_search,
    "database_query": mock_database_query,
    "file_reader": mock_file_reader,
}

print("Mock 工具已加载:")
for name in MOCK_TOOLS:
    print(f"  - {name}")

In [ ]:
# ===== Mock LLM =====

class MockLLM:
    """Mock LLM —— 模拟语言模型的响应。

    这个 Mock 对象满足 BaseAgent 对 llm 参数的要求。
    在实际使用中，SimpleAgent 主要通过规则来做决策，
    所以 MockLLM 只是一个占位符，确保接口一致。
    """

    def __init__(self, name: str = "MockGPT"):
        self.name = name
        self.call_count = 0

    def __call__(self, prompt: str) -> str:
        """使 MockLLM 可调用。"""
        return self.generate(prompt)

    def generate(self, prompt: str) -> str:
        """生成模拟响应。"""
        self.call_count += 1
        return f"[{self.name}] 这是对 '{prompt[:50]}...' 的第 {self.call_count} 次模拟响应"


mock_llm = MockLLM()
print(f"Mock LLM 已创建: {mock_llm.name}")
print(f"测试调用: {mock_llm.generate('你好')}")

## 6. 运行演示

现在将 SimpleAgent、MockLLM 和 MockTools 组合起来，运行完整的 Agent 控制循环。

In [ ]:
def run_demo(title: str, task: str):
    """运行一次 Agent 演示。

    Args:
        title: 演示标题
        task: 用户任务
    """
    print("\n" + "=" * 60)
    print(f"  {title}")
    print(f"  任务: {task}")
    print("=" * 60)

    # 创建 Agent
    agent = SimpleAgent(
        llm=mock_llm,
        tools=MOCK_TOOLS,
        max_iterations=10
    )

    # 运行
    context = agent.run(task)

    # 输出结果
    print("\n" + "=" * 60)
    print(f"  执行完成!")
    print(f"  总步数: {context.current_step}")
    print(f"  最终状态: {context.state}")
    print(f"  工具调用: {len(context.observations)} 次")
    print("=" * 60)

    if agent.final_answer:
        print(f"\n[最终回答]\n{agent.final_answer}")

    # 打印状态转移
    agent.print_state_history()

    return agent


# === 演示 1: 天气查询 ===
agent1 = run_demo(
    title="演示 1: 天气查询",
    task="查询北京的天气并给出穿衣建议"
)

In [ ]:
# === 演示 2: 数学计算 ===
agent2 = run_demo(
    title="演示 2: 数学计算",
    task="计算 25 * 4 + 10 的结果"
)

In [ ]:
# === 演示 3: 信息搜索 ===
agent3 = run_demo(
    title="演示 3: 信息搜索",
    task="查询故宫门票价格"
)

In [ ]:
# === 演示 4: 数据库查询 ===
agent4 = run_demo(
    title="演示 4: 数据库查询",
    task="查询技术部员工的工资信息"
)

## 7. max_iterations 防护机制

`max_iterations` 是防止 Agent 无限循环的关键安全机制。
下面演示当 Agent 陷入"永远觉得信息不够"的情况时，max_iterations 如何保护系统。

In [ ]:
# 创建一个总是需要继续的 Agent（模拟无限循环场景）

class IndecisiveAgent(SimpleAgent):
    """一个总是觉得信息不够的 Agent —— 用于演示无限循环防护。"""

    def reflect(self, context: AgentContext, result: dict) -> bool:
        """总是返回 True（需要继续），直到被 max_iterations 截断。"""
        results = result.get("results", [])
        if results:
            print(f"    反思: 获得了结果，但我还想获取更多信息...")
        else:
            print(f"    反思: 没有结果，继续尝试...")
        return True  # 永远继续


print("\n" + "=" * 60)
print("  演示: max_iterations 无限循环防护")
print("  Agent 被设置为永远不自我终止")
print("=" * 60)

indecisive_agent = IndecisiveAgent(
    llm=mock_llm,
    tools=MOCK_TOOLS,
    max_iterations=5  # 设置一个较小的值以快速触发
)

context = indecisive_agent.run(task="查询北京天气")

print(f"\n最终步数: {context.current_step}")
print(f"最终状态: {context.state}")
print(f"被强制终止: {context.has_reached_limit()}")
indecisive_agent.print_state_history()

## 8. 控制循环可视化

用 Colorama 给控制循环加上颜色，让状态转移更加直观。

In [ ]:
# 可视化版本的 Agent（带彩色输出）

# ANSI 颜色码（不依赖第三方库）
class Colors:
    """ANSI 终端颜色。"""
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    END = '\033[0m'


class VisualAgent(SimpleAgent):
    """带彩色可视化的 Agent。"""

    STATE_COLORS = {
        AgentState.PERCEIVING: Colors.BLUE,
        AgentState.PLANNING: Colors.CYAN,
        AgentState.ACTING: Colors.GREEN,
        AgentState.REFLECTING: Colors.YELLOW,
        AgentState.FINISHED: Colors.BOLD + Colors.GREEN,
        AgentState.ERROR: Colors.RED,
    }

    def _transition_to(self, new_state: AgentState):
        """带颜色的状态转移输出。"""
        super()._transition_to(new_state)
        color = self.STATE_COLORS.get(new_state, '')
        step = self.context.current_step if self.context else 0
        print(f"{color}  [{new_state}] Agent 进入 '{str(new_state)}' 状态 {Colors.END}")

    def run(self, task: str) -> AgentContext:
        """运行并打印流程摘要。"""
        print(f"\n{Colors.BOLD}{Colors.HEADER}")
        print(f"  ╔══════════════════════════════════════════════╗")
        print(f"  ║        Agent 控制循环 - 可视化演示          ║")
        print(f"  ╚══════════════════════════════════════════════╝")
        print(f"{Colors.END}")
        print(f"  {Colors.BOLD}任务:{Colors.END} {task}")
        print(f"  {Colors.BOLD}最大迭代:{Colors.END} {self.max_iterations}")
        print()

        context = super().run(task)

        print(f"\n{Colors.BOLD}{'=' * 50}{Colors.END}")
        print(f"{Colors.BOLD}  执行摘要{Colors.END}")
        print(f"{Colors.BOLD}{'=' * 50}{Colors.END}")
        print(f"  总步数: {context.current_step}")
        print(f"  工具调用: {len(context.observations)}")
        print(f"  对话轮数: {len(context.history)}")

        if self.final_answer:
            print(f"\n  {Colors.GREEN}[最终回答]{Colors.END}")
            print(f"  {self.final_answer[:200]}")

        return context


# 运行可视化演示
visual_agent = VisualAgent(
    llm=mock_llm,
    tools=MOCK_TOOLS,
    max_iterations=10
)
visual_agent.run("查询北京天气并给出穿衣建议")

## 9. 总结

### 本 Notebook 的核心要点

1. **Agent 架构**：Agent = LLM + Tools + Memory + Control Loop
   - LLM：提供推理和决策能力
   - Tools：扩展 Agent 的能力边界
   - Memory：存储上下文和历史信息
   - Control Loop：协调各组件的执行流程

2. **状态机**：IDLE → PERCEIVING → PLANNING → ACTING → REFLECTING → FINISHED
   - 清晰的状态定义让 Agent 行为可预测
   - 状态转移记录便于调试和监控

3. **控制循环**：
   - `max_iterations` 是关键的安全机制
   - 每个阶段都可以独立测试和优化
   - 异常处理防止单个步骤的错误导致整个 Agent 崩溃

4. **AgentContext**：
   - 承载所有执行状态
   - 在阶段之间传递信息
   - 提供执行历史的完整记录

### 下一步

现在你已经理解了 Agent 的控制循环骨架，接下来：
- **02-react-agent-from-scratch.py**：从零实现完整的 ReAct Agent
- **03-function-calling-deep-dive.ipynb**：深入工具调用的高级技巧